# Project ASTRA: Gravitational Lensing Extragalactic Sandbox

This interactive notebook demonstrates the **Caskade** Pydantic initialization logic 
and **JAX/Equinox** operations natively integrated with `diffrax` Neural ODEs.

In [ ]:
!pip install -q jax jaxlib equinox optax diffrax pydantic pyyaml astropy scipy pytest matplotlib

In [ ]:
import jax
import jax.numpy as jnp
import equinox as eqx

# 1. Strictly Typed Configuration Setup natively validating inputs
from src.utils.config import CaskadePipelineConfig, LensModelConfig, SourceConfig, SimulationConfig

lens = LensModelConfig(model_type='NFW', mass_solar=1e12, concentration=10.0)
source = SourceConfig(position=(0.5, 0.5), radius_arcsec=0.2)
sim = SimulationConfig(grid_extent=5.0, grid_resolution=256)

config = CaskadePipelineConfig(
    lens=lens,
    source=source,
    simulation=sim
)

print("✅ Pipeline Initialized cleanly:\n", config.model_dump_json(indent=2))

### 2. Differentiable Ray Tracing mapped cleanly to Config

In [ ]:
from src.optics.ray_tracing import ray_trace
from src.lens_models.mass_profiles import NFWProfile
from src.lens_models.lens_system import LensSystem

sys = LensSystem(config.lens.z_l, config.source.z_s)
nfw = NFWProfile(float(config.lens.mass_solar), float(config.lens.concentration), sys)

results = ray_trace(
    source_position=config.source.position,
    lens_model=nfw,
    grid_extent=config.simulation.grid_extent,
    grid_resolution=config.simulation.grid_resolution
)

print(f"Ray tracing complete. Yielding {len(results['image_positions'])} independent highly-magnified anomalies.")